# Lab 6 — Software optimization + MCU feasibility
Measure graph optimization configurations and estimate a hypothetical MCU memory budget. Do not confuse a desktop ONNX file with flash layout on a real MCU.


In [ ]:
!pip -q install onnxruntime onnx numpy


In [ ]:
from google.colab import files
uploaded=files.upload() # select gesture_model.onnx; optionally select a session CSV
import os, numpy as np, onnx, onnxruntime as ort, time, pandas as pd
model_path='gesture_model.onnx'
assert os.path.exists(model_path), 'Please upload gesture_model.onnx'
onnx.checker.check_model(onnx.load(model_path))
options=ort.SessionOptions();options.intra_op_num_threads=1
sess=ort.InferenceSession(model_path,options,providers=['CPUExecutionProvider'])
features=np.zeros((1,42),dtype=np.float32)
print('Input',sess.get_inputs()[0].name,sess.get_inputs()[0].shape)
print('Logits',sess.run(None,{sess.get_inputs()[0].name:features})[0])


In [ ]:
levels={'disabled':ort.GraphOptimizationLevel.ORT_DISABLE_ALL,'basic':ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,'extended':ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,'all':ort.GraphOptimizationLevel.ORT_ENABLE_ALL}
def bench_one(s,x,n=300):
 feed={s.get_inputs()[0].name:x}
 for _ in range(20):s.run(None,feed)
 ts=[]
 for _ in range(n):
  t=time.perf_counter();s.run(None,feed);ts.append((time.perf_counter()-t)*1000)
 return float(np.median(ts)),float(np.percentile(ts,95))
for label,level in levels.items():
 opt=ort.SessionOptions();opt.graph_optimization_level=level;opt.intra_op_num_threads=1
 # Save the optimized graph to see which operators ORT fused or removed at this level.
 opt.optimized_model_filepath=f'optimized_{label}.onnx'
 candidate=ort.InferenceSession(model_path,opt,providers=['CPUExecutionProvider'])
 ops=[n.op_type for n in onnx.load(opt.optimized_model_filepath).graph.node]
 print(label,'median/p95 ms:',bench_one(candidate,features),'| operators:',ops)
# TODO: 1) Why might optimizations not help this tiny model? 2) Compare against browser end-to-end FPS.


In [ ]:
import onnx
from onnx import numpy_helper
model=onnx.load(model_path)
parameter_values=sum(len(t.raw_data) for t in model.graph.initializer)
parameter_count=sum(int(np.prod(t.dims)) for t in model.graph.initializer)
print('ONNX initializer count:',parameter_count,'raw-data bytes:',parameter_values)
weights=[numpy_helper.to_array(t) for t in model.graph.initializer if len(t.dims)==2]
weight_count=sum(w.size for w in weights);bias_count=parameter_count-weight_count
FLASH_KB=256;SRAM_KB=64
print('FP32 parameter bytes estimate:',parameter_count*4)
# INT8 deployments keep biases as int32 (4 bytes) and add a scale/zero-point per layer.
print('INT8 parameter bytes estimate (int8 weights + int32 biases):',weight_count+bias_count*4)
print('Hypothetical MCU Flash:',FLASH_KB,'KB; SRAM:',SRAM_KB,'KB')
# TODO: compute peak activations (batch=1). A layer's input and output buffers are live at the same time,
# so peak = max over layers of (inputs + outputs); for 42->64->32->4 that is 42+64 = 106 floats = 424 bytes FP32.
# NOTE: full peak RAM also includes runtime arena, temporary buffers, firmware, stack, and allocations.
